# matmul-back-transpose-pair — ex2: batched matmul_back — transpose(-1, -2) on the OTHER input

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `matmul-back-transpose-pair`. Running the final beacon cell reports progress against the `Backprop: matmul_back transpose pair` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: matmul_back transpose pair` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`matmul-back-transpose-pair`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "matmul-back-transpose-pair"
DD_SUBTOPIC = "Backprop: matmul_back transpose pair"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Batched `matmul_back`: `(B, m, k) @ (B, k, n)` — quick refresher

ex1 covered the 2-D transpose pair. The deeper facet is what happens when there's a batch axis on the LEFT: every formula stays the same, but `.T` becomes `.transpose(-1, -2)` so it only swaps the LAST two axes.

```
x: (B, m, k),  y: (B, k, n),  out: (B, m, n)

dL/dx must be (B, m, k)  →  grad_out @ y.transpose(-1, -2)
                            : (B, m, n) @ (B, n, k) = (B, m, k)  ✓
dL/dy must be (B, k, n)  →  x.transpose(-1, -2) @ grad_out
                            : (B, k, m) @ (B, m, n) = (B, k, n)  ✓
```

Why `.transpose(-1, -2)` not `.T`. On a `(B, m, k)` tensor, `.T` reverses ALL axes → `(k, m, B)`, which is the wrong shape. `transpose(-1, -2)` swaps only the last two, leaving the batch alone — which is what every framework's batched matmul backward does.

### Exercise 2 — batched matmul_back — transpose(-1, -2) on the OTHER input

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the matmul transpose-pair pattern to batched inputs (B, m, k) @ (B, k, n) using transpose(-1, -2) on the partner input.
> Keywords: matmul, batched, transpose, broadcasting
> ```

**KCs targeted:** `matmul-backward-pattern`, `arg-position-back-functions`

Implement TWO back fns for batched matmul `out = x @ y` where:
- `x` shape `(B, m, k)`
- `y` shape `(B, k, n)`
- `out` shape `(B, m, n)`

**1. `bmm_back0(grad_out, out, x, y)`** — gradient w.r.t. `x`.
   - Target shape `(B, m, k)`.
   - `grad_out @ y.transpose(-1, -2)`: `(B, m, n) @ (B, n, k) = (B, m, k)`. ✓

**2. `bmm_back1(grad_out, out, x, y)`** — gradient w.r.t. `y`.
   - Target shape `(B, k, n)`.
   - `x.transpose(-1, -2) @ grad_out`: `(B, k, m) @ (B, m, n) = (B, k, n)`. ✓

**Why `.transpose(-1, -2)` and not `.T`.** On a 3-D tensor, `.T` reverses ALL axes — `(B, m, k)` would become `(k, m, B)`, which is the wrong shape AND scrambles the batch axis. `transpose(-1, -2)` swaps only the last two; the batch axis stays put.

Use `@` / `t.matmul`. Return tensors with the correct shapes. No autograd. The ex1 2-D transpose-pair is the `B=1` special case.

In [ ]:
def bmm_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # (B, m, n) @ (B, n, k) = (B, m, k).
    return grad_out @ y.transpose(-1, -2)


def bmm_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # (B, k, m) @ (B, m, n) = (B, k, n).
    return x.transpose(-1, -2) @ grad_out


<details><summary>Solution</summary>

```python
def bmm_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # (B, m, n) @ (B, n, k) = (B, m, k).
    return grad_out @ y.transpose(-1, -2)


def bmm_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # (B, k, m) @ (B, m, n) = (B, k, n).
    return x.transpose(-1, -2) @ grad_out
```

**Why `.transpose(-1, -2)` is the batched analog of `.T`.** On a 2-D tensor `.T` swaps the only two axes — but on a 3-D tensor it reverses ALL axes. `transpose(-1, -2)` is the explicit form: swap EXACTLY the matmul axes, leaving every leading axis alone. This is what `torch.matmul`'s own backward uses.

**Per-batch independence is the structural invariant.** Batched matmul is just B independent 2-D matmuls run in parallel. The test that stacks per-batch results pins this down — if you accidentally summed across the batch axis, that assertion would fail.

**Where this generalizes.** Attention's `Q @ K.T` and `attn @ V` are both batched matmuls — over heads, over batch, sometimes over sequence prefixes. The transpose-the-OTHER-input rule survives every such generalization.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()